# Data Fusion Pipeline: OpenCorporates + Semantic Scholar

## Project Overview
This notebook documents the data fusion pipeline developed during my PhD research at INSA Strasbourg. The system links **OpenCorporates company data** with **Semantic Scholar academic publications** to analyze the research foundations of energy startups.

### Objectives
1. Extract startup information from OpenCorporates API
2. Link company officers to their academic publications via Semantic Scholar API
3. Filter and score articles by interdisciplinary level
4. Create a novel dataset for innovation analysis in renewable energy

### Data Sources
| Source | Purpose | Key Fields |
|--------|---------|------------|
| **OpenCorporates** | Company identification | Company name, officers, incorporation date, industry codes |
| **Semantic Scholar** | Publication linkage | Author names, article titles, fields of study, abstracts |

### Pipeline Steps
1. **Company Extraction**: Query OpenCorporates for energy startups (2020-2025)
2. **Officer Matching**: Extract officer names and affiliations
3. **Publication Retrieval**: Search Semantic Scholar for matching authors
4. **Filtering**: Keep only energy-relevant research fields
5. **Scoring**: Apply interdisciplinary scoring system
6. **Analysis**: Generate insights and visualizations

### Dataset Summary (Latest Run)
- **Startups identified**: ~27,000 companies
- **Startups linked to publications**: ~15,000 companies
- **Total articles processed**: ~178,207 relevant publications
- **Time period**: 2020-2025 (companies), all years (publications)


In [ ]:
import requests
# Your API key
api_key = ''

# Base URL for the OpenCorporates API - to update when new version is released
base_url = 'https://api.opencorporates.com/v0.4/companies/search'

In [ ]:
# API CALL TO FIND ALL ENERGY COMPANIES FROM GB, FR, NO, North America, and EU general

from pandas import json_normalize
import requests
from tqdm import tqdm
from datetime import datetime

# OpenCorporates API Key (Replace with your actual key)
API_KEY = ''

# API Endpoint & Parameters
BASE_URL = "https://api.opencorporates.com/v0.4/companies/search"

# EXAMPLE : INDUSTRY_CODE_LIST = ["uk_sic_2007-351","isic_4-3510","eu_nace_2-351","no_sic_2007-351","us_naics_2017-2211"]
# All above codes correpond to energy production, transport, and storage
INDUSTRY_CODE_LIST = ["","","","",""]

#change date range
INCORPORATION_DATE = "YYYY-MM-DD:YYYY-MM-DD"

#variable to store results before writing to an outfile
data_list = []

for INDUSTRY_CODE in tqdm(INDUSTRY_CODE_LIST):
    page = 1
    while True:
        
        # Construct API Request URL
        # No nonprofits, no inactive; can change
        params = {
            "api_token": API_KEY,
            "incorporation_date": INCORPORATION_DATE,
            "industry_codes": INDUSTRY_CODE,
            "inactive": False,
            "nonprofit": False,
            "per_page": 100,
            "page": page
        
        }

        # Make API Request
        response = requests.get(BASE_URL, params=params)
        #convert to JSON format, new variable to reflect the new structure
        data = response.json()
        json_data = data

        if json_data["results"]["companies"]:
            for row in json_data["results"]["companies"]:
                data_list.append(row)
                #Page count, can be removed
                print(f"Data found, page #{page}")
        else:
            break
        
        if page == 100:
            break
        page += 1

if data_list:
    df = json_normalize(data_list)
    df.to_csv(f"PATH/TO/DIRECTORY{datetime.today().strftime('%Y-%m-%d')}.csv", encoding="utf8", index=False)
    print("CSV saved successfully.")
    print(f"There are {len(df)} entries")
else:
    print("No valid data to save.")

In [ ]:
#Check structure of DF
df.head()

In [ ]:

#quick filtering of inactive companies and duplicates

# To update with other undesirable status tags to filter out inactive companies
df_filtered = df[~df['company.current_status'].isin(["Liquidation", 'Dissolved', "Cessée", "None"])]

#Drop duplicates that were classed under the same industry code
df_filtered = df_filtered.drop_duplicates(subset=['company.jurisdiction_code', 'company.company_number', 'company.incorporation_date'])

print(len(df_filtered))

#Filtering by creation date, optional step
#df_new_startups = df_filtered[(df_filtered['company.incorporation_date']> "2021-01-01") & (df_filtered['company.incorporation_date'] < "2025-08-01")]
#df[(df['date'] > '2013-01-01') & (df['date'] < '2013-02-01')]
df_new_startups = df_filtered
len(df_new_startups)


In [ ]:

"""OpenCorporates requires jurisdiction codes & \
    company numbers in their registry in order to \
    request the full entry for each company
"""

from pandas import json_normalize
import requests
from tqdm import tqdm

# Set up token and URL
api_token = 'oa3LofUKOArFSWYYRp8f'
#search_url = f'https://api.opencorporates.com/v0.4/companies/{num}/officers'

data_list = []
jurisdiction_codes = list(df_new_startups['company.jurisdiction_code'])
company_numbers = list(df_new_startups['company.company_number'])


for code, num in tqdm(list(zip(jurisdiction_codes, company_numbers))): 
    
    
    data_params = {
        'api_token':api_token,
       
    }
    
    search_url = f'https://api.opencorporates.com/v0.4/companies/{code}/{num}'
    try:
        # Perform and process get request
        r = requests.get(url = search_url, params=data_params)
        data = r.json()
        json_data = data
        data_list.append(json_data)
         
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e} ")
        continue
        
      
if data_list:
    df_full_info = json_normalize(data_list)
    
    df_full_info.to_csv(f"PATH/TO/DIRECTORY{datetime.today().strftime('%Y-%m-%d')}.csv", encoding="utf8", index=False)
    print("CSV saved successfully.")
else:
    print("No valid data to save.")


In [ ]:
#Extracting list of officers from each company
from tqdm import tqdm
import ast
tqdm.pandas()

# creating officer list to go through easily
def officer_list(results_company_officers):
    """Creating a separate column for the list of officers in each company

    Args:
        results_company_officers (json): Full json response from OpenCorp

    Returns:
        List: list of all officers found in each company
    """
    
    officers_list = []
    results_company_officers = ast.literal_eval(results_company_officers)
    if results_company_officers and type(results_company_officers) == list:
        for officer in results_company_officers:
                
                officers_list.append(officer['officer']['name'])
    else:
        officers_list.append('No officers found')
    return officers_list
            
df_full_info['officer_list'] = df_full_info['results.company.officers'].progress_apply(officer_list)


In [ ]:
# counting rows that contain officer names/that don't contain officer names

no_officers_count = df_full_info['officer_list'].apply(lambda x: 'No officers found' in x).sum()
print(f"Rows with no officers found: {no_officers_count}")

officers = df_full_info['officer_list'].to_list()

officers = [list for list in officers if not 'No officers found' in list]
print(f"Rows with officers found: {len(officers)}")

In [ ]:
import requests
import time
import random

#should increase max attempts to 5 or more
def fetch_papers_by_author_name(officer_names, api_key='', max_attempts=10, author_limit=10):
    """Semantic Scholar integration - requesting articles written \
        by officers of startups based on name

    Args:
        officer_names (List): List of officers in each company
        api_key (str, optional): api key for Semantic Scholar. Defaults to ''.
        max_attempts (int, optional): variable for when no api key is used. \
            Progressive backing off is used to avoid IP address blocking. Defaults to 10.
        author_limit (int, optional): Limit for authors with the same name, \
            useful for filtering out results for common names. Semantic Scholar will \
                take the most relevant results first. Defaults to 10.

    Returns:
        json: json response from Semantic Scholar
    """
    all_author_articles = []
    headers = {"x-api-key": api_key, "Content-Type": "application/json"}
    base_url = "https://api.semanticscholar.org/graph/v1"

    # Fields to request for each paper
    fields = "title,abstract,year,fieldsOfStudy,citationCount,referenceCount,authors"

    for name in officer_names:
        author_ids = []
        # Step 1: Search for author(s) by name
        for _ in range(max_attempts):
            try:
                search_url = f"{base_url}/author/search"
                params = {"query": name, "limit": author_limit}
                search_response = requests.get(search_url, headers=headers, params=params)
                
                if search_response.status_code == 200:
                    data = search_response.json()
                    author_ids = [a['authorId'] for a in data.get("data", [])]
                    
                    #adding name to avoid searching through multiple columns
                    author_ids = {name:author_ids}
                    
                    break  # exit retry loop
                elif search_response.status_code == 429:
                    time.sleep(random.randint(5, 10))
                else:
                    break
            except Exception as e:
                print(f"Exception during author search for '{name}': {e}")
                break

        if not author_ids:
            all_author_articles.append({'name': name, 'author_id': None, 'papers': []})
            continue

        # Step 2: For each matching author ID, fetch their papers
        for author_name, author_ids in author_ids.items():
            
            for author_id in author_ids:
                for attempt in range(max_attempts):
                    try:
                        papers_url = f"{base_url}/author/{author_id}/papers"
                        params = {"fields": fields, "limit": 100}
                        papers_response = requests.get(papers_url, headers=headers, params=params)

                        if papers_response.status_code == 200:
                            papers_data = papers_response.json().get("data", [])
                            all_author_articles.append({
                                'name': name,
                                'open_corp_name': author_name,
                                'author_id': author_id,
                                'papers': papers_data
                            })
                            print("success")
                            break
                        elif papers_response.status_code == 429:
                            #print("Rate limited on papers fetch — sleeping...")
                            time.sleep(random.randint(5, 10))
                        else:
                            #print(f"Failed paper fetch for author ID {author_id} — status {papers_response.status_code}")
                            break
                    except Exception as e:
                        #print(f"Exception fetching papers for author ID {author_id}: {e}")
                        break
                # to avoid hitting 1 request/second rate limit
                time.sleep(1.1)  

    return all_author_articles



In [ ]:
"""Final enriched data - final column will contain officer/author names 
   and articles that they have published
"""
df['officer_articles'] = df['officer_list'].progress_apply(fetch_papers_by_author_name)

    

In [ ]:
#Saving final data

df.to_csv("PATH/TO/SAVE.csv")


## Transdisciplinarity Scoring System

To assess the level of interdisciplinary integration in scientific articles linked to energy startups, we developed a scoring system inspired by TRIZ inventive levels but adapted for **field-of-study diversity** rather than contradiction analysis.

### Category Mapping

First, we grouped Semantic Scholar’s `fieldsOfStudy` into six **macro-categories**:

| Macro-category | Included Fields |
|----------------|-----------------|
| **Applied Sciences** | Engineering, Materials Science |
| **Formal Sciences** | Mathematics, Computer Science |
| **Natural Sciences** | Physics, Chemistry, Environmental Science, Geology, Geography |
| **Business** | Business, Economics |
| **Medicine** | Medicine |
| **Outside Fields** | Psychology, Sociology, Art, History, Philosophy, Political Science, Biology (non-technical) |

### Scoring Logic

Each article is assigned a score based on the **combination of fields** present:

| Score | Description | Example |
|-------|-------------|---------|
| **1** | Business-only (monodisciplinary) | Business, Economics |
| **1.5** | Monodisciplinary within a **technical** macro-category | Physics (only), Engineering (only) |
| **2** | One technical field + Business | Engineering + Business |
| **3** | Two subfields **within the same** macro-category | Chemistry + Physics (both Natural Sciences) |
| **4** | Two fields from **different technical** macro-categories | Engineering + Mathematics |
| **5** | **Conceptually distant** combinations (technical + outside fields, or applied + natural sciences) | Engineering + Art, Physics + Sociology |

### Filtering Rules

We **excluded** articles that were:
- Monodisciplinary and in non-technical/non-business domains (e.g., Art only)
- Combinations of only Business + Outside Fields
- Medicine as the sole or dominant field

### Purpose of Scoring

This scoring system allows us to:
- Quantify **disciplinary diversity** in startup-linked research
- Identify articles with **high cross-domain integration** (Scores 3–5)
- Filter for **innovation-signaling** publications without manual review

### Result Distribution (from our dataset)

| Score | Number of Articles | % of Corpus |
|-------|-------------------|----------------------|
| 0 | 10,591 | 5.94% |
| 1 | 15,557 | 8.73% |
| 1.5 | 150,585 | 84.49% |
| 2 | 551 | 0.31% |
| 3 | 3,910 | 2.19% |
| 4 | 7,013 | 3.93% |
| **Total** | **178,207** | **100.00%** |

> **Note:** Level 5 (Applied + Natural Sciences) yielded **0 results** in our corpus—consistent with the rarity of such breakthrough combinations.

